In [54]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
import xgboost as xgb
import optuna

In [55]:
df = pd.read_csv('../data/processed/final_dataset.csv')
df.head()

,sleep_quality,training_intensity,training_load,fatigue_index,body_temperature,heart_rate,hydration_level,muscle_activity,ground_reaction_force,ambient_temperature,recovery_time_hours,sport_type_Other,sport_type_Soccer,sport_type_Track,gender_Male
0,5.466370,5.678475,548.417962,45.028687,37.048668,72.304051,76.481477,10.000000,1859.705455,17.603241,13.6,0,0,0,0
1,6.899090,6.411926,538.043815,42.254717,36.929536,40.000000,72.363938,310.293156,1852.051883,25.107144,13.9,0,0,0,0
2,9.574152,6.643242,647.784339,61.870056,36.334841,76.174609,100.000000,158.659094,1654.115836,27.894979,13.0,0,0,0,0
3,7.614297,6.194427,336.553431,50.268845,37.244474,100.543724,86.172473,107.079298,1727.371018,27.886378,13.4,0,0,0,0
4,5.631979,5.835804,496.076352,29.233575,36.510766,78.001235,83.195889,164.347133,1642.459351,22.812019,17.0,0,0,0,0


In [56]:
df.columns

Index(['sleep_quality', 'training_intensity', 'training_load', 'fatigue_index',
       'body_temperature', 'heart_rate', 'hydration_level', 'muscle_activity',
       'ground_reaction_force', 'ambient_temperature', 'recovery_time_hours',
       'sport_type_Other', 'sport_type_Soccer', 'sport_type_Track',
       'gender_Male'],
      dtype='object')

In [58]:
numerical_features = [
       'sleep_quality', 
       'training_intensity', 
       'training_load', 
       'fatigue_index',
       'body_temperature',
       'heart_rate',   
       'hydration_level', 
       'muscle_activity',
       'ground_reaction_force', 
       'ambient_temperature',
]

categorical_encoded = [
    'gender_Male', 
    'sport_type_Other', 
    'sport_type_Soccer', 
    'sport_type_Track',
]  

features = numerical_features + categorical_encoded
target = 'recovery_time_hours'

X = df[features]
y = df[target]

Y_MAX_CONSTANT = y.max() + 1

In [59]:
# Train-Test Split
# Fixed: Changed 'test_test_split' to 'test_size' - this is the correct parameter name
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
y_train_transformed = np.log1p(Y_MAX_CONSTANT - y_train)
y_test_transformed = np.log1p(Y_MAX_CONSTANT - y_test)

In [60]:
# Scale numerical features
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_test_scaled[numerical_features] = scaler.transform(X_test[numerical_features])

In [61]:
baseline_model = LinearRegression()
baseline_model.fit(X_train_scaled, y_train_transformed)

baseline_preds_transformed = baseline_model.predict(X_test_scaled)
baseline_preds_real = Y_MAX_CONSTANT - np.expm1(baseline_preds_transformed)

print("--- Baseline Linear Regression Performance (Option A Optimized) ---")
print(f"R² Score : {r2_score(y_test, baseline_preds_real):.4f}")
print(f"MAPE     : {mean_absolute_percentage_error(y_test, baseline_preds_real) * 100:.2f}%")

--- Baseline Linear Regression Performance (Option A Optimized) ---
R² Score : 0.5508
MAPE     : 13.89%


In [62]:
models = {
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": xgb.XGBRegressor(random_state=42)
}

print("--- Multi-Model Initial Evaluation (Option A Optimized) ---")
for name, model in models.items():
    # 1. Fit the model using the stable, transformed symmetric target variable
    model.fit(X_train_scaled, y_train_transformed)
    
    # 2. Predict in the transformed log-space
    preds_transformed = model.predict(X_test_scaled)
    
    # 3. Mathematically invert predictions back to true recovery hours
    preds_real = Y_MAX_CONSTANT - np.expm1(preds_transformed)
    
    # 4. Calculate metrics using real-world hours vs. real-world test data
    r2 = r2_score(y_test, preds_real)
    mape = mean_absolute_percentage_error(y_test, preds_real) * 100
    
    print(f"{name:18} -> R²: {r2:.4f} | MAPE: {mape:.2f}%")

# %% [markdown]

--- Multi-Model Initial Evaluation (Option A Optimized) ---
Ridge              -> R²: 0.5508 | MAPE: 13.89%
Lasso              -> R²: -0.0192 | MAPE: 22.01%
Random Forest      -> R²: 0.6188 | MAPE: 11.80%
Gradient Boosting  -> R²: 0.6275 | MAPE: 11.68%
XGBoost            -> R²: 0.5902 | MAPE: 12.16%


In [63]:
def objective_option_a(trial):
    # Hyperparameters for XGBoost
    xgb_params = {
        'n_estimators': trial.suggest_int('xgb_n_estimators', 50, 300),
        'max_depth': trial.suggest_int('xgb_max_depth', 3, 5),
        'learning_rate': trial.suggest_float('xgb_learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('xgb_subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('xgb_colsample_bytree', 0.6, 1.0),
        'random_state': 42
    }
    
    # Hyperparameters for Gradient Boosting
    gb_params = {
        'n_estimators': trial.suggest_int('gb_n_estimators', 50, 300),
        'max_depth': trial.suggest_int('gb_max_depth', 3, 5),
        'learning_rate': trial.suggest_float('gb_learning_rate', 0.01, 0.2, log=True),
        'random_state': 42
    }
    
    # Initialize models
    model_xgb = xgb.XGBRegressor(**xgb_params)
    model_gb = GradientBoostingRegressor(**gb_params)
    model_rf = RandomForestRegressor(
        n_estimators=trial.suggest_int('rf_n_estimators', 100, 300), 
        max_depth=trial.suggest_int('rf_max_depth', 3, 5), 
        random_state=42
    )
    
    # Ensemble Weights to balance individual estimators
    w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
    w_gb = trial.suggest_float('w_gb', 0.0, 1.0)
    w_rf = trial.suggest_float('w_rf', 0.0, 1.0)
    
    # 1. Fit sub-models on the stable, symmetric target data
    model_xgb.fit(X_train_scaled, y_train_transformed)
    model_gb.fit(X_train_scaled, y_train_transformed)
    model_rf.fit(X_train_scaled, y_train_transformed)
    
    # 2. Extract raw ensemble predictions in transformed log-space
    total_w = w_xgb + w_gb + w_rf
    preds_transformed = ((w_xgb * model_xgb.predict(X_test_scaled)) + 
                         (w_gb * model_gb.predict(X_test_scaled)) + 
                         (w_rf * model_rf.predict(X_test_scaled))) / total_w
    
    # 3. Mathematically invert ensemble predictions back to original recovery hours
    preds_real = Y_MAX_CONSTANT - np.expm1(preds_transformed)
    
    # 4. Objective function evaluates real hours performance against raw validation data
    return r2_score(y_test, preds_real)

# Run Optuna Study to maximize true hours R2 score
study = optuna.create_study(direction='maximize')
study.optimize(objective_option_a, n_trials=30)

print("[SUCCESS] Tuning complete.")
print(f"Best Trial R² Score: {study.best_value:.4f}")

[I 2026-07-10 00:36:15,547] A new study created in memory with name: no-name-f3bf7fd0-63eb-4c50-9897-fd490964b8c3
[I 2026-07-10 00:36:25,628] Trial 0 finished with value: 0.6235809560214975 and parameters: {'xgb_n_estimators': 113, 'xgb_max_depth': 3, 'xgb_learning_rate': 0.11865214782260446, 'xgb_subsample': 0.9925615729443159, 'xgb_colsample_bytree': 0.620798094049012, 'gb_n_estimators': 202, 'gb_max_depth': 4, 'gb_learning_rate': 0.016602358410871904, 'rf_n_estimators': 224, 'rf_max_depth': 3, 'w_xgb': 0.9085825678254309, 'w_gb': 0.7911066877136295, 'w_rf': 0.2986750626325225}. Best is trial 0 with value: 0.6235809560214975.
[I 2026-07-10 00:36:36,704] Trial 1 finished with value: 0.5921098643316398 and parameters: {'xgb_n_estimators': 154, 'xgb_max_depth': 5, 'xgb_learning_rate': 0.010241054116394612, 'xgb_subsample': 0.8532811715579639, 'xgb_colsample_bytree': 0.916627271106256, 'gb_n_estimators': 186, 'gb_max_depth': 3, 'gb_learning_rate': 0.023159737185683003, 'rf_n_estimators':

[SUCCESS] Tuning complete.
Best Trial R² Score: 0.6345


In [64]:
best_params = study.best_params

# 2. Reconstruct best estimators using optimized hyperparameters
champion_xgb = xgb.XGBRegressor(
    n_estimators=best_params['xgb_n_estimators'],
    max_depth=best_params['xgb_max_depth'],
    learning_rate=best_params['xgb_learning_rate'],
    subsample=best_params['xgb_subsample'],
    colsample_bytree=best_params['xgb_colsample_bytree'],
    random_state=42
)

champion_gb = GradientBoostingRegressor(
    n_estimators=best_params['gb_n_estimators'],
    max_depth=best_params['gb_max_depth'],
    learning_rate=best_params['gb_learning_rate'],
    random_state=42
)

champion_rf = RandomForestRegressor(
    n_estimators=best_params['rf_n_estimators'],
    max_depth=best_params['rf_max_depth'],
    random_state=42
)

# 3. Extract and normalize raw blending weights for human interpretation
raw_weights = [best_params['w_xgb'], best_params['w_gb'], best_params['w_rf']]
total_weight = sum(raw_weights)
normalized_weights = [w / total_weight for w in raw_weights]

# 4. Initialize VotingRegressor using optimized weights
champion_ensemble = VotingRegressor(
    estimators=[('xgb', champion_xgb), ('gb', champion_gb), ('rf', champion_rf)],
    weights=raw_weights
)

# 5. Fit the ensemble on the stable, transformed symmetric training target
champion_ensemble.fit(X_train_scaled, y_train_transformed)

# 6. Predict in log-reflected space and mathematically invert back to original recovery hours
final_preds_transformed = champion_ensemble.predict(X_test_scaled)
final_preds_real = Y_MAX_CONSTANT - np.expm1(final_preds_transformed)

# --- Print Weights and Performance Metrics ---
print("--- Optimized Ensemble Weights ---")
print(f"XGBoost Weight           : {best_params['w_xgb']:.4f} (Normalized: {normalized_weights[0]*100:.1f}%)")
print(f"Gradient Boosting Weight : {best_params['w_gb']:.4f} (Normalized: {normalized_weights[1]*100:.1f}%)")
print(f"Random Forest Weight     : {best_params['w_rf']:.4f} (Normalized: {normalized_weights[2]*100:.1f}%)")
print("-" * 43)

print("--- Final Champion Ensemble Performance (Option A Optimized) ---")
print(f"R² Score : {r2_score(y_test, final_preds_real):.4f}")
print(f"MAPE     : {mean_absolute_percentage_error(y_test, final_preds_real) * 100:.2f}%")

--- Optimized Ensemble Weights ---
XGBoost Weight           : 0.8647 (Normalized: 72.7%)
Gradient Boosting Weight : 0.2106 (Normalized: 17.7%)
Random Forest Weight     : 0.1140 (Normalized: 9.6%)
-------------------------------------------
--- Final Champion Ensemble Performance (Option A Optimized) ---
R² Score : 0.6345
MAPE     : 11.52%


In [65]:
import joblib

# Define export paths for the artifacts
model_filename = "../models/champion_recovery_ensemble.pkl"
features_filename = "../models/ensemble_feature_columns.pkl"

# 1. Save the fitted VotingRegressor ensemble
joblib.dump(champion_ensemble, model_filename)

# 2. Save the explicit list of columns X_train_scaled saw during training
# This prevents shape mismatches during downstream inference or SHAP evaluations
joblib.dump(list(X.columns), features_filename)

print(f"[SUCCESS] Model artifact exported as: '{model_filename}'")
print(f"[SUCCESS] Feature tracking reference exported as: '{features_filename}'")

[SUCCESS] Model artifact exported as: '../models/champion_recovery_ensemble.pkl'
[SUCCESS] Feature tracking reference exported as: '../models/ensemble_feature_columns.pkl'
